# Étude de cas 5
## Jalon 1 : Estimer la durée de vie restante
**Plan** : 0. Préparation · Q1 référence et erreur selon la RUL · Q2 normalisation  · Q3 perte asymétrique · Q4 fenêtre de k cycles · Vérifications théoriques · Test · Page de décisions.

**Protocole** : réglages sur la **validation** (15 pompes, découpage par pompe) ; le **test** n'est utilisé qu'une fois, à la fin (`EVAL_TEST`).

## 0. Préparation

In [1]:
# ============================================================================
# 0. PRÉPARATION — imports, graine, chemins et chargement des données
# ============================================================================
import platform, sklearn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from pathlib import Path
# Crée les dossiers de sortie s'ils n'existent pas déjà.
Path("figures").mkdir(exist_ok=True); Path("models").mkdir(exist_ok=True)
def sauver_fig(nom):
    # Sauvegarde la figure courante dans figures/ avec une résolution lisible.
    plt.savefig(f"figures/{nom}.png", dpi=150, bbox_inches="tight")

# Graine globale : rend les tirages aléatoires reproductibles.
SEED = 42
RUL_MAX = 125          # Plafond de la RUL : au-delà, les capteurs ne montrent plus de dégradation.
EVAL_TEST = True       # Le test a déjà été évalué une seule fois ; ne pas le réexécuter sans raison.
SEEDS = (0, 1, 2)      # Trois graines pour distinguer un effet réel de la variabilité aléatoire.
tf.keras.utils.set_random_seed(SEED)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| tensorflow", tf.__version__, "| sklearn", sklearn.__version__)

# Charge la flotte A : train, validation et test sont déjà séparés par pompe.
A = np.load("data/uc5_flotteA.npz")
# Les capteurs sont anonymisés dans le fichier : c0 à c7.
CAPTEURS = [f"c{i}" for i in range(8)]

# Journal d'expériences : chaque essai est ajouté ici puis exporté en CSV.
JOURNAL = []

def log(essai, **res):
    # Ajoute une ligne de résultat au journal, avec le numéro de jalon.
    JOURNAL.append(dict(jalon=1, essai=essai, **res))


ModuleNotFoundError: No module named 'sklearn'

**Données et normalisation.** Les capteurs dépendent du régime de charge (60/80/100 %) : on centre-réduit **par régime** avec les statistiques des cycles **sains** du train (RUL > 150). Même statistiques pour val et test (pas de fuite). Entrées du MLP : 8 capteurs normalisés + la charge = **9 entrées**. Cible : `min(RUL, 125) / 125`.

In [ ]:
# ============================================================================
# DONNÉES, NORMALISATION ET CIBLE
# ============================================================================
def vers_df(s):
    # Transforme les tableaux NumPy du .npz en DataFrame pandas pour un split donné.
    d = pd.DataFrame(A["capteurs_" + s], columns=CAPTEURS)
    # A est le dictionnaire chargé depuis le fichier .npz.
    d["pompe"], d["cycle"], d["charge"] = A["pompe_" + s], A["cycle_" + s], A["charge_" + s]
    # La RUL et le mode n'existent que pour les historiques complets (train et validation).
    if "rul_" + s in A.files:
        d["rul"], d["mode"] = A["rul_" + s], A["mode_" + s]
    return d

# Construit les trois DataFrames et vérifie que chaque split est trié par pompe puis cycle.
TA = {s: vers_df(s) for s in ["train", "val", "test"]}
for s in TA:
    assert TA[s].sort_values(["pompe", "cycle"]).index.equals(TA[s].index)

# Statistiques de normalisation calculées uniquement sur les cycles sains du train (RUL > 150).
sains = TA["train"][TA["train"].rul > 150]
# mu_r / sd_r : moyennes et écarts-types par régime de charge.
mu_r, sd_r = sains.groupby("charge")[CAPTEURS].mean(), sains.groupby("charge")[CAPTEURS].std()
# mu_g / sd_g : statistiques globales, utilisées seulement pour l'ablation de la Q2.
mu_g, sd_g = TA["train"][CAPTEURS].mean(), TA["train"][CAPTEURS].std()

def normaliser(df):
    # Centre et réduit chaque capteur avec les statistiques de son régime de charge.
    out = df.copy()
    out[CAPTEURS] = (df[CAPTEURS].values - mu_r.loc[df.charge].values) / sd_r.loc[df.charge].values
    return out

def normaliser_global(df):
    # Variante d'ablation : une seule moyenne et un seul écart-type pour tous les régimes.
    out = df.copy()
    out[CAPTEURS] = (df[CAPTEURS] - mu_g) / sd_g
    return out

# NA : normalisation par régime (retenue) ; NG : normalisation globale (ablation).
NA = {s: normaliser(TA[s]) for s in TA}
NG = {s: normaliser_global(TA[s]) for s in TA}

def xy(df):
    # Entrées : 8 capteurs normalisés + la charge = 9 variables.
    X = df[CAPTEURS + ["charge"]].values.astype("float32")
    # Cible bornée dans [0, 1] : min(RUL, 125) / 125.
    y = (np.minimum(df.rul.values, RUL_MAX) / RUL_MAX).astype("float32") if "rul" in df else None
    return X, y

def score_nasa(y, yhat):
    # Score asymétrique NASA : d = prédit - réel, en cycles.
    # d < 0 (alerte précoce) coûte moins cher que d > 0 (panne manquée).
    d = yhat - y
    return np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)

# Affiche le nombre de cycles par split pour contrôler le chargement.
print({s: len(TA[s]) for s in TA})


**Métriques** (en cycles, sur la validation, tous les cycles d'historiques complets) :
- `rmse` sur la RUL plafonnée ; `score30` = 30 × score moyen par cycle (équivalent pour 30 pompes, à comparer à la cible 1 200 du test) ;
- `anticipation` : part des cycles à RUL ≤ 30 où la RUL prédite est < 30 (pompe « signalée ») ; `fausses` : part des cycles à RUL > 60 signalés à tort ;
- `biais` : moyenne de (prédit − réel).

*Approximation* : le sujet mesure anticipation et fausses alertes **par pompe** sur le test ; ici on les mesure par cycle sur des historiques complets.

In [ ]:
# ============================================================================
# OUTILS D'ENTRAÎNEMENT ET D'ÉVALUATION
# ============================================================================
def construire_mlp(d, perte="mse"):
    # MLP de référence : d entrées -> 64 -> 32 -> 1, sortie sigmoïde dans [0, 1].
    m = keras.Sequential([layers.Input((d,)), layers.Dense(64, activation="relu"),
                          layers.Dense(32, activation="relu"), layers.Dense(1, activation="sigmoid")])
    if perte == "mse":
        # Perte MSE classique, sans écrêtage du gradient.
        m.compile(keras.optimizers.Adam(1e-3), "mse")
    else:
        # Perte personnalisée : clipnorm limite l'explosion due aux exponentielles.
        m.compile(keras.optimizers.Adam(1e-3, clipnorm=1.0), perte)
    return m

def entrainer(Xtr, ytr, Xva, yva, seed, perte="mse"):
    # Fixe la graine avant la création du modèle pour rendre l'essai reproductible.
    tf.keras.utils.set_random_seed(seed)
    m = construire_mlp(Xtr.shape[1], perte)
    # Arrêt anticipé sur la validation : on restaure les poids de la meilleure époque.
    m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=128, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)])
    return m

def evaluer(m, Xv, yv):
    # Repasse des valeurs normalisées [0, 1] aux cycles réels.
    y, yh = RUL_MAX * yv, RUL_MAX * m.predict(Xv, verbose=0).ravel()
    return dict(rmse=float(np.sqrt(np.mean((yh - y) ** 2))),
                # score30 : score moyen par cycle ramené à l'échelle de 30 pompes.
                score30=float(30 * score_nasa(y, yh).mean()),
                # Anticipation : cycles à RUL réelle <= 30 effectivement signalés.
                anticipation=float((yh[y <= 30] < 30).mean()),
                # Fausses alertes : cycles encore loin de la panne (RUL > 60) signalés à tort.
                fausses=float((yh[y > 60] < 30).mean()),
                # Biais : moyenne de (prédit - réel) ; positif = RUL surestimée.
                biais=float((yh - y).mean()))

def resume(rows, par):
    # Agrège les essais par condition et calcule moyenne et écart-type.
    t = pd.DataFrame(rows).groupby(par, sort=False).agg(["mean", "std"])
    return t.drop(columns="seed").round(3)


## Q1. Reproduire la référence et tracer l'erreur selon la RUL réelle
MLP de référence : 9 entrées → 64 → 32 → 1 (sigmoïde), perte MSE, dernier cycle observé seulement.

In [ ]:
# ============================================================================
# Q1 — MLP DE RÉFÉRENCE
# ============================================================================
# Prépare les entrées et les cibles normalisées du train et de la validation.
Xtr, ytr = xy(NA["train"]); Xva, yva = xy(NA["val"])
rows = []
# Entraîne un modèle par graine et journalise ses métriques de validation.
for seed in SEEDS:
    m = entrainer(Xtr, ytr, Xva, yva, seed)
    r = evaluer(m, Xva, yva); rows.append(dict(seed=seed, **r)); log(f"reference_seed{seed}", **r)
# Moyenne et écart-type des métriques sur les trois graines.
# (le `if False` neutralise l'ancien appel à resume() ; seul le calcul direct est exécuté)
print(resume(rows, lambda i: "reference") if False else pd.DataFrame(rows).drop(columns="seed").agg(["mean", "std"]).round(3))
# Modèle de référence sauvegardé : graine globale 42, distincte des graines 0, 1 et 2 ci-dessus.
mlp = entrainer(Xtr, ytr, Xva, yva, SEED)
mlp.save("models/jalon1_mlp_reference.keras")


In [ ]:
# ============================================================================
# Q1 — ERREUR SELON LA RUL RÉELLE
# ============================================================================
# Écart en cycles : prédit - réel ; positif = RUL surestimée (panne potentiellement manquée).
ecart = RUL_MAX * (mlp.predict(Xva, verbose=0).ravel() - yva)
plt.scatter(RUL_MAX * yva, ecart, s=2, alpha=0.3)
plt.axhline(0, color="r", ls=":")
plt.xlabel("RUL réelle (plafonnée)"); plt.ylabel("RUL prédite - RUL réelle")
sauver_fig("q1_ecart_vs_rul"); plt.show()

# Découpe la RUL en tranches pour quantifier le biais et la dispersion.
bornes = [0, 15, 30, 45, 60, 90, 125, 126]
tr = pd.cut(RUL_MAX * yva, bornes, right=False)
print(pd.DataFrame({"ecart": ecart, "tranche": tr}).groupby("tranche", observed=True).ecart
      .agg(biais="mean", ecart_type="std", n="size").round(1))


**Lecture.** Biais **négatif** = RUL sous-estimée (alerte trop tôt) ; **positif** = RUL surestimée (panne imprévue, le sens dangereux). Le modèle de référence **surestime fortement** la RUL entre 30 et 90 cycles de la panne (biais d'environ +13 à +26 cycles) : c'est la zone où le défaut commence à peine à se voir, et c'est là que la dispersion est la plus forte (écart-type ≈ 25-30). Près de la panne (RUL < 30) l'erreur est faible. Dans la zone plafonnée (RUL = 125) le biais est négatif (≈ −10) : la sigmoïde sature avant 1 et ne peut pas atteindre le plafond.

## Q2. Normalisation globale vs par régime
Même MLP, mêmes graines ; seule la normalisation change. Hypothèse : sans normalisation par régime, la charge (qui décale les capteurs) est confondue avec la dégradation.

In [ ]:
# ============================================================================
# Q2 — ABLATION : NORMALISATION GLOBALE VS PAR RÉGIME
# ============================================================================
# NG utilise des statistiques globales ; NA utilise une normalisation par régime.
Xg, yg = xy(NG["train"]); Xgv, ygv = xy(NG["val"])
rows = []
for seed in SEEDS:
    # Essai avec normalisation globale.
    r = evaluer(entrainer(Xg, yg, Xgv, ygv, seed), Xgv, ygv)
    rows.append(dict(norm="globale", seed=seed, **r)); log(f"norm_globale_seed{seed}", **r)
    # Essai avec normalisation par régime, sur les mêmes graines.
    r = evaluer(entrainer(Xtr, ytr, Xva, yva, seed), Xva, yva)
    rows.append(dict(norm="par régime", seed=seed, **r)); log(f"norm_regime_seed{seed}", **r)
# Tableau comparatif : moyenne et écart-type pour chaque normalisation.
res2 = pd.DataFrame(rows)
tab2 = res2.groupby("norm", sort=False)[["rmse", "score30", "anticipation", "fausses"]].agg(["mean", "std"]).round(3)
print(tab2)


**Lecture.** La normalisation par régime fait passer la RMSE de 19,13 à 18,92 et le score de 787 à 767, soit des gains de 0,20 cycle et 20 points. Ces écarts sont du même ordre que la variabilité entre graines (écart-type 0,16 en RMSE, 34 à 80 en score) : **l'ablation ne démontre pas que la normalisation par régime améliore la prédiction**. L'hypothèse de départ (la charge décale les capteurs et se confond avec la dégradation) reste plausible physiquement, mais elle n'est pas mesurable sur ces 15 pompes. On conserve néanmoins la normalisation par régime : elle ne coûte rien et elle est indispensable au transfert vers le type B (jalon 3).

## Q3. Perte asymétrique inspirée du score
Avec d = RUL prédite − RUL réelle (en cycles) : `e^{−d/13} − 1` si d < 0 (alerte trop tôt, gentille) et `e^{d/10} − 1` sinon (panne ratée, sévère). On l'utilise comme perte (différentiable) ; `clipnorm` évite l'explosion des gradients due à l'exponentielle.

In [ ]:
# ============================================================================
# Q3 — PERTE ASYMÉTRIQUE INSPIRÉE DU SCORE NASA
# ============================================================================
from keras import ops
def perte_asym(y_true, y_pred):
    # Écart en cycles : surestimer la RUL (d > 0) sera plus pénalisé que la sous-estimer.
    d = RUL_MAX * (y_pred - y_true)
    # Version TensorFlow conservée en commentaire pour référence.
    #return tf.reduce_mean(tf.where(d < 0, tf.exp(-d / 13.0) - 1.0, tf.exp(d / 10.0) - 1.0))
    return ops.mean(ops.where(d < 0, ops.exp(-d / 13.0) - 1.0, ops.exp(d / 10.0) - 1.0))

rows = []
# Compare MSE et perte asymétrique sur les mêmes graines.
for seed in SEEDS:
    for nom, perte in [("MSE", "mse"), ("asymétrique", perte_asym)]:
        r = evaluer(entrainer(Xtr, ytr, Xva, yva, seed, perte), Xva, yva)
        rows.append(dict(perte=nom, seed=seed, **r)); log(f"perte_{nom}_seed{seed}", **r)
# Tableau comparatif : RMSE, score, anticipation, fausses alertes et biais.
res3 = pd.DataFrame(rows)
tab3 = res3.groupby("perte", sort=False)[["rmse", "score30", "anticipation", "fausses", "biais"]].agg(["mean", "std"]).round(3)
print(tab3)


**Lecture.** La perte asymétrique **déplace le biais** de +1,13 à −12,31 cycles : le modèle prédit plutôt trop tôt. Le score est divisé par 2,7 (≈ 767 → 286), mais le RMSE augmente (≈ 18,9 → 23,7). L'anticipation ne change presque pas (≈ 0,80 → 0,83) et les fausses alertes restent nulles pour les deux pertes : à ce seuil de 30 cycles, la métrique « fausses alertes » par cycle ne départage pas les pertes. Le choix repose donc sur le compromis score / RMSE.

## Q4. Fenêtre de k cycles aplatie, k ∈ {1, 5, 15, 30}
Entrées : les k derniers cycles (k × 9 valeurs ; début d'historique : 1er cycle répété). **Bruit** = écart-type de la variation cycle à cycle de la prédiction, mesuré là où la RUL réelle décroît de 1 par cycle (RUL ≤ 125) : une prédiction parfaitement lisse aurait un écart-type nul.

In [ ]:
# ============================================================================
# Q4 — FENÊTRE DE k CYCLES APLATIE
# ============================================================================
def fenetres_aplaties(df, k):
    # Construit, pour chaque pompe, une entrée de k cycles consécutifs aplatis.
    X, y, dernier = [], [], []
    for _, g in df.sort_values(["pompe", "cycle"]).groupby("pompe"):
        v = g[CAPTEURS + ["charge"]].values.astype("float32")
        # Début d'historique : répète le premier cycle pour compléter la fenêtre.
        v = np.vstack([np.repeat(v[:1], k - 1, axis=0), v])
        # Une ligne par cycle : les k derniers cycles mis bout à bout (k x 9 valeurs).
        X.append(np.stack([v[i:i + k].ravel() for i in range(len(g))]))
        # Indique le dernier cycle observé de chaque pompe (utile pour le test).
        dernier.append(np.arange(len(g)) == len(g) - 1)
        if "rul" in g:
            y.append(np.minimum(g.rul.values, RUL_MAX) / RUL_MAX)
    y = np.concatenate(y).astype("float32") if y else None
    return np.concatenate(X), y, np.concatenate(dernier)

def bruit(m, Xv):
    # Mesure l'instabilité de la prédiction : écart-type des variations cycle à cycle.
    yh = RUL_MAX * m.predict(Xv, verbose=0).ravel()
    pompe, rul = NA["val"].pompe.values, NA["val"].rul.values
    s = []
    for p in np.unique(pompe):
        # On ne garde que la zone où la RUL réelle est dans la partie utile (<= 125).
        i = np.where((pompe == p) & (rul <= RUL_MAX))[0]
        s.append(np.diff(yh[i]).std())
    return float(np.mean(s))

rows = []
# Teste quatre tailles de fenêtre, avec trois graines chacune.
for k in (1, 5, 15, 30):
    Xk, yk, _ = fenetres_aplaties(NA["train"], k); Xkv, ykv, _ = fenetres_aplaties(NA["val"], k)
    for seed in SEEDS:
        m = entrainer(Xk, yk, Xkv, ykv, seed)
        r = evaluer(m, Xkv, ykv); r["bruit"] = bruit(m, Xkv)
        rows.append(dict(k=k, seed=seed, **r)); log(f"fenetre_k{k}_seed{seed}", **{a: round(b, 3) for a, b in r.items()})
# Tableau comparatif : chaque métrique en moyenne et écart-type selon k.
res4 = pd.DataFrame(rows)
tab4 = res4.groupby("k")[["rmse", "score30", "anticipation", "fausses", "bruit"]].agg(["mean", "std"]).round(3)
print(tab4)


In [ ]:
# ============================================================================
# Q4 — VISUALISATION k = 1 VS k = 30 SUR UNE POMPE DE VALIDATION
# ============================================================================
# Choisit la première pompe de validation et sélectionne ses cycles.
p0 = NA["val"].pompe.unique()[0]; msk = (NA["val"].pompe == p0).values
for k in (1, 30):
    # Réentraîne un modèle pour chaque fenêtre avec la graine globale.
    Xk, yk, _ = fenetres_aplaties(NA["train"], k); Xkv, ykv, _ = fenetres_aplaties(NA["val"], k)
    m = entrainer(Xk, yk, Xkv, ykv, SEED)
    # Trace la RUL prédite pour cette pompe.
    plt.plot(RUL_MAX * m.predict(Xkv, verbose=0).ravel()[msk], label=f"k={k}", alpha=0.8)
# Trace la RUL réelle plafonnée en pointillés noirs.
plt.plot(RUL_MAX * ykv[msk], "k--", label="réel (plafonné)")
plt.xlabel("cycle"); plt.ylabel("RUL"); plt.legend()
sauver_fig("q4_rul_k1_vs_k30"); plt.show()


## Modèle retenu (réglages sur la validation)
Critère : **score asymétrique** (coût des erreurs) ; les fausses alertes valent ≈ 0 pour tous les essais, donc elles ne servent ici que de contrôle. On retient la perte de plus faible `score30` moyen (évaluée à k = 1), puis le k de plus faible `score30` moyen (évalué sous MSE), puis on réentraîne le modèle final (graine fixée).

*À noter sur la fenêtre* : k = 5 réduit le bruit (≈ 10 → 6,8) et le score ; au-delà (k = 15, 30) le bruit remonte (≈ 7.8 à 8,6) alors que l'anticipation s'améliore (jusqu'à ≈ 0,84 pour k = 30) : plus de contexte aide à détecter la dégradation, mais l'écart de bruit entre k = 5, 15 et 30 reste modeste.

*Protocole de sélection (limite).* La perte est choisie dans `tab3`, entraînée à **k = 1** ; la fenêtre est choisie dans `tab4`, entraînée avec **MSE**. La combinaison retenue (perte asymétrique, k = 5) n'est donc évaluée dans aucun des deux tableaux : elle est réentraînée à part et mesure RMSE 21,94, score 256,5, anticipation 0,806. L'interaction perte × fenêtre n'a pas été explorée ; en particulier, MSE à k = 30 obtient la meilleure anticipation (0,849), supérieure à celle du modèle retenu (0,806).

In [ ]:
# ============================================================================
# MODÈLE RETENU — SÉLECTION SUR LA VALIDATION
# ============================================================================
# Choisit la perte de plus faible score30 moyen (tableau Q3, évalué à k = 1).
PERTE_RETENUE = tab3[("score30", "mean")].idxmin()
# Choisit la fenêtre de plus faible score30 moyen (tableau Q4, évalué sous MSE).
K_RETENU = int(tab4[("score30", "mean")].idxmin())
print("perte retenue :", PERTE_RETENUE, "| k retenu :", K_RETENU)
# Convertit le nom de la perte en objet utilisable par Keras.
perte_obj = "mse" if PERTE_RETENUE == "MSE" else perte_asym
# Réentraîne la combinaison perte x fenêtre avec la graine globale.
Xk, yk, _ = fenetres_aplaties(NA["train"], K_RETENU); Xkv, ykv, _ = fenetres_aplaties(NA["val"], K_RETENU)
final = entrainer(Xk, yk, Xkv, ykv, SEED, perte_obj)
final.save("models/jalon1_final.keras")
# Évalue le modèle final sur la validation et journalise le résultat.
r_final = evaluer(final, Xkv, ykv); log("modele_final", perte=PERTE_RETENUE, k=K_RETENU, **{a: round(b, 3) for a, b in r_final.items()})
print({a: round(b, 3) for a, b in r_final.items()})
print("cibles finales (test) : RMSE <= 23 | score <= 1200 | anticipation >= 0,90 | fausses alertes <= 0,10")


## Vérifications théoriques (à la main, puis vérifiées par le code)

**1. Paramètres du MLP (9 → 64 → 32 → 1).** Couche 1 : 9·64 + 64 = 640 ; couche 2 : 64·32 + 32 = 2 080 ; sortie : 32·1 + 1 = 33. **Total = 2 753.**

**2. Contribution au score** (d = prédit − réel) : d = −10 → e^{10/13} − 1 ≈ **1,16** ; d = +10 → e^{1} − 1 ≈ **1,72** ; d = −30 → e^{30/13} − 1 ≈ **9,05** ; d = +30 → e^{3} − 1 ≈ **19,09**.
*Asymétrie* : surestimer la RUL (panne ratée, d > 0) coûte plus cher que la sous-estimer (alerte précoce), et l'écart croît avec |d| (×1,5 à ±10, ×2,1 à ±30). *Conséquence* : une perte symétrique (MSE) traite pareil les deux erreurs ; une perte asymétrique pousse le modèle à prédire plutôt trop bas.

In [ ]:
# ============================================================================
# VÉRIFICATIONS THÉORIQUES
# ============================================================================
# Vérifie le nombre de paramètres : (9*64+64) + (64*32+32) + (32*1+1) = 2753.
assert construire_mlp(9).count_params() == 9*64+64 + 64*32+32 + 32+1 == 2753
# Vérifie les contributions du score NASA pour quelques écarts typiques.
for d in (-10, 10, -30, 30):
    print(f"d = {d:+d} -> contribution = {score_nasa(0.0, float(d)):.2f}")


## Test (à exécuter UNE seule fois, à la fin)
`EVAL_TEST` vaut déjà `True` dans la première cellule ; le test a été exécuté **une seule fois**, à la fin. Pour le réexécuter, relancer uniquement cette cellule. Mesures **par pompe**, au dernier cycle observé (30 pompes) ; anticipation : pompes à RUL finale ≤ 30 ; fausses alertes : pompes à RUL finale > 60.

In [ ]:
# ============================================================================
# TEST FINAL — 30 POMPES, DERNIER CYCLE OBSERVÉ
# ============================================================================
if EVAL_TEST:
    # Construit les fenêtres du test et repère le dernier cycle de chaque pompe.
    Xt, _, dern = fenetres_aplaties(NA["test"], K_RETENU)
    pompes = TA["test"].sort_values(["pompe", "cycle"]).pompe.values[dern]
    # Récupère la RUL finale réelle de chaque pompe de test.
    rul_fin = pd.Series(A["rul_fin_test"], index=A["pompe_id_test"]).loc[pompes].values
    # Prédit au dernier cycle observé, puis ramène la sortie en cycles.
    yh = RUL_MAX * final.predict(Xt[dern], verbose=0).ravel()
    y = np.minimum(rul_fin, RUL_MAX)
    # Métriques par pompe : RMSE, score total, anticipation et fausses alertes.
    print("RMSE test :", round(float(np.sqrt(np.mean((yh - y) ** 2))), 2),
          "| score :", round(float(score_nasa(y, yh).sum()), 0),
          "| anticipation :", round(float((yh[rul_fin <= 30] < 30).mean()), 3),
          "| fausses alertes :", round(float((yh[rul_fin > 60] < 30).mean()), 3))
else:
    print("Test non évalué (EVAL_TEST = False).")


## Page de décisions

**Tableau des résultats** : `tab2` (normalisation), `tab3` (perte), `tab4` (fenêtre) et modèle final ci-dessus ; journal dans `journal_jalon1.csv`.

**Normalisation (Q2).** L'ablation ne montre pas d'effet démontré : globale 19,13 / 787 contre par régime 18,92 / 767, soit un écart de 0,20 RMSE et 20 de score, du même ordre que l'écart-type entre graines (0,16 en RMSE, 34 à 80 en score). On conserve la normalisation par régime, cohérente avec les jalons suivants, sans prétendre qu'elle améliore la RMSE.

**Plafonnement de la RUL à 125.** Loin de la panne, les capteurs ne portent aucun signe de dégradation (figure du sujet : courbes plates avant ~100 cycles) : distinguer RUL = 200 de RUL = 300 est impossible et ferait apprendre du bruit. Le plafond donne une cible bornée, cohérente avec le cahier des charges (RMSE défini sur la RUL plafonnée), et concentre l'effort sur la zone utile.

**Choix de la perte.** Le critère métier est asymétrique : une panne imprévue coûte plus qu'un remplacement précoce. On compare MSE et perte asymétrique sur RMSE, score, anticipation et fausses alertes ; le choix retenu est celui du score le plus bas (voir « Modèle retenu »), en vérifiant que les fausses alertes restent acceptables.

**Activation de sortie : sigmoïde.** La cible `min(RUL,125)/125` est dans [0, 1] : la sigmoïde garantit des prédictions bornées (pas de RUL négative ni au-delà du plafond). Contrepartie : saturation aux extrémités, où les gradients sont faibles.

**Écart aux cibles.** Les cibles décrivent le système *final* (test, par pompe). Sur la **validation** (tous les cycles), le modèle final atteint : RMSE ≈ 21,9 (cible ≤ 23 ✓), score pour 30 pompes ≈ 257 (cible ≤ 1 200 ✓), fausses alertes ≈ 0,00 (cible ≤ 0,10 ✓), mais **anticipation ≈ 0,81 (cible ≥ 0,90 ✗)** : c'est l'écart principal. Cette anticipation n'est que marginalement supérieure à celle du MLP MSE de référence (0,800) : la perte asymétrique achète le score (767 → 257) au prix du RMSE (18,9 → 21,9) sans améliorer nettement la détection précoce.
Sur le **test** (30 pompes, dernier cycle), le modèle atteint RMSE 20,83 (≤ 23 ✓), score 344 (≤ 1 200 ✓), anticipation 0,818 (≥ 0,90 ✗) et 0,00 de fausses alertes (≤ 0,10 ✓). L'écart à la cible d'anticipation est donc confirmé sur le test, pas seulement en validation. Causes plausibles : un seul cycle bruité (ou 5) ; entre 30 et 60 cycles de la panne le défaut est peu visible et le modèle surestime la RUL ; absence des spectrogrammes (jalon 2) et d'un modèle dédié au type B (jalon 3). *Limites* : données simulées, 15 pompes de validation, mesures par cycle (et non par pompe) en validation ; les chiffres exacts peuvent varier légèrement selon la machine.

In [ ]:
# ============================================================================
# EXPORT DU JOURNAL D'EXPÉRIENCES
# ============================================================================
# Écrit une ligne par essai dans journal_jalon1.csv (à la racine du notebook).
pd.DataFrame(JOURNAL).to_csv("journal_jalon1.csv", index=False)
print(len(JOURNAL), "essais journalisés -> journal_jalon1.csv")
